# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Descrição
Este notebook tem como objetivo analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Etapas de Pré-processamento
Este notebook descreve as etapas de pré-processamento necessárias para analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Necessidades de Processamento
- **Identificar o ponto de divisão** entre as reflexões do Terço e as reflexões do Dia.
- **Separar os textos** em reflexões do Terço e reflexões do Dia.
- **Remover as seções** marcadas com a tag `[Música]`.
- **Identificar e remover orações oficiais** da Igreja Católica.
- **Extrair e documentar músicas cantadas**, incluindo o nome e o autor de cada música.
- **Considerar o momento do Rosário** em que cada ensinamento foi apresentado, relacionando-o ao terço e ao mistério meditado naquele momento.

## 1.0. Importação de Bibliotecas


In [ ]:
from langchain.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# MODEL = "deepseek-r1:8b"
# MODEL = "gemma3:12b"
# MODEL = "gemma3:27b-it-qat"
# MODEL = "phi4:14b"
# MODEL = "mistral-small3.1"
# MODEL = "DarkIdol-Llama-3.1-8B-Instruct-1.2-Uncensored:latest"

MODEL = "gemini-2.0-flash"

# Providers: azure_openai, deepseek, bedrock, openai, fireworks, xai, mistralai,
# ollama, google_anthropic_vertex, cohere, google_vertexai, perplexity,
# azure_ai, google_genai, ibm, bedrock_converse, groq, together, anthropic, huggingface
MODEL_PROVIDER = "google_genai"

## 2.0. Processamento de texto

In [ ]:
def remove_starting_tabs(text):
    """
    Remove starting tabs from the text.
    """
    lines = text.split("\n")
    cleaned_lines = [line.lstrip("\t") for line in lines]
    return "\n".join(cleaned_lines)

In [ ]:
system_message = f"""
    Você é um especialista na fé católica.
    Você receberá um texto que é uma transcrição de um vídeo do YouTube.

    O vídeo é Santo Rosário, rezado diariamente ao vivo pelo Frei Gilson durante a um dia da quaresma do ano de 2025.
    Durante o vídeo, o Frei Gilson reza o Santo Rosário e faz uma série de reflexões e ensinamentos sobre a fé católica.

    O vídeo é dividido em duas partes bem distintas:
    1. A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    2. A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    Você vai se atentar a informações prestadas apenas durante a primeira parte do vídeo (o Santo Rosário),
    e vai recuperar essas informações devidamente formatadas e organizadas,
    a fim de que o usuário possa ter um resumo de todo ensinamento que foi transmitido.
    
    Você vai trazer somente informações que estejam presentes na primeira parte do vídeo: o Santo Rosário.
    Você não pode trazer informações que estejam presentes apenas na segunda parte do vídeo (a reflexão),
    a menos que também o Santo Rosário do dia traga esse conhecimento.
    
    Você não precisa explicar o funcionamento do Santo Rosário, 
    a menos que essa informação seja relevante para algum ensinamento que você trouxer.
    
    Você não precisa trazer transcricões de oracões católicas conhecidas, como o Pai Nosso, a Ave Maria ou o Credo.
    
    Ignore as marcações de tempo presentes na transcrição.
    Não invente informações, não faça suposições, não faça inferências, somente traga informações presentes na transcrição.
    
    Você vai responder apenas com um relatório markdown, exatamente com as mesmas seções e formatações que estão no exemplo abaixo:
    
    ```markdown
    # Relatório do Santo Rosário
    
    ## Temática principal
    
    <2 a 3 parágrafos de informações e ensinamentos sobre a temática principal dos ensinamentos trazidos junto a este Santo Rosário>
    
    ## Temáticas secundárias
    
    ### <temática secundária 1>
    
    <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 1>
    
    ### <temática secundária 2>
    
    <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 2>
    
    ...
    
    ### <temática secundária N>
    
    <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária N>
    
    ## Versículos citados na transcrição
    
    <Identifique cuidadosamente todos os versículos que foram citados na transmissão do Santo Rosário>
    <Você vai trazer a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson>
    <Se houver necessidade, identifique o momento do Santo ROsário em que o versículo foi citado (Qual terço, qual mistério, etc.)>
    <Você não vai deixar de fora nenhum versículo que tenha sido citado>
    Ex.:
    
    - Se o versículo João 3, 16 foi citado, você vai trazer:
    
    ```
    (João 3, 16): <Transcrição integral do versículo>
    
    Ensinamentos: <Ensinamentos que foram transmitidos por aquele versículo>
    ```
    
    ## Músicas
    
    <Todas as músicas que foram citadas durante o Santo Rosário>
    
    Formato:
    
    ```
    <Nome da música> - <Artista>
    ```
    
    ## Eventos de Agenda
    
    <Todos os eventos de agenda que foram citados durante o Santo Rosário>
    
    Formato.:
    ```
    <Título> - <data e horário> - <local>
    <Descrição do evento>
    ```
    
    Por favor, não traga seções vazias, nem seções que não estejam no exemplo acima.
    Nã responda nada além do relatório markdown exatamente como o exemplo acima.
"""

## 3.0. Leitura dos Arquivos

In [ ]:
import os
from urllib import response

from tqdm import tqdm
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)

for i in tqdm(range(1, 41), desc="Processando arquivos", unit="arquivo"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                print(f"\n\nArquivo: {titulo_source}")

                messages = [
                    {"role": "system", "content": remove_starting_tabs(system_message)},
                    {"role": "user", "content": conteudo},
                ]
                    
                response = model.invoke(messages).content
                
                with open(
                    f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                ) as f:
                    f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")